<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Exercises_XP_VDB_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [1]:
# Installation des versions les plus récentes
# On force numpy < 2.1 pour assurer la compatibilité avec pandas et numba dans Colab
%pip install -U pydantic pydantic-settings
%pip install -U faiss-cpu chromadb
%pip install -U "numpy<2.1" sentence-transformers transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 81.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.0
    Uninstalling numpy-2.5.0:
      Successfully uninstalled numpy-2.5.0


In [1]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
# Adaptation pour Pydantic V2 : Settings est maintenant géré différemment ou via pydantic_settings
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display
os.makedirs('cache', exist_ok=True)

## 🌟 Exercise 1 · Data loading and preparation

In [15]:
import pandas as pd
import os

data_path = 'labelled_newscatcher_dataset.csv'

if os.path.exists(data_path):
    try:
        # Lecture du fichier local importé par l'utilisateur
        # On tente avec le séparateur ';' qui est courant pour ce dataset
        pdf = pd.read_csv(data_path, sep=';', on_bad_lines='skip')

        # Si le dataframe est mal chargé (une seule colonne), on tente avec ','
        if len(pdf.columns) <= 1:
            pdf = pd.read_csv(data_path, sep=',', on_bad_lines='skip')

        pdf['id'] = range(len(pdf))
        pdf_subset = pdf.head(1000).copy()
        pdf_to_index = pdf_subset

        print(f"Succès ! Dataset local chargé : {len(pdf_subset)} lignes.")
        display(pdf_subset.head())
    except Exception as e:
        print(f"Erreur lors de la lecture du fichier local : {e}")
else:
    print(f"Erreur : Le fichier '{data_path}' est introuvable. Veuillez vous assurer de l'avoir uploadé dans le dossier 'content'.")

Succès ! Dataset local chargé : 1000 lignes.


,topic,link,domain,published_date,title,lang,id
0,SCIENCE,https://www.eurekalert.org/pub_releases/2020-0...,eurekalert.org,2020-08-06 13:59:45,A closer look at water-splitting's solar fuel ...,en,0
1,SCIENCE,https://www.pulse.ng/news/world/an-irresistibl...,pulse.ng,2020-08-12 15:14:19,"An irresistible scent makes locusts swarm, stu...",en,1
2,SCIENCE,https://www.express.co.uk/news/science/1322607...,express.co.uk,2020-08-13 21:01:00,Artificial intelligence warning: AI will know ...,en,2
3,SCIENCE,https://www.ndtv.com/world-news/glaciers-could...,ndtv.com,2020-08-03 22:18:26,Glaciers Could Have Sculpted Mars Valleys: Study,en,3
4,SCIENCE,https://www.thesun.ie/tech/5742187/perseid-met...,thesun.ie,2020-08-12 19:54:36,Perseid meteor shower 2020: What time and how ...,en,4


## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [18]:
def example_create_fn(idx: int, text: str) -> InputExample:
    return InputExample(guid=str(idx), texts=[text], label=0.0)

# Correction : Utilisation du pdf_subset chargé localement
if 'pdf_subset' in globals():
    faiss_train_examples = [example_create_fn(row['id'], row['title']) for _, row in pdf_subset.iterrows()]
    print(f"Nombre d'exemples créés : {len(faiss_train_examples)}")
    print("Aperçu des deux premiers exemples :")
    for ex in faiss_train_examples[:2]:
        print(f" ID: {ex.guid}, Text: {ex.texts}")
else:
    print("Erreur : pdf_subset n'est pas défini. Relancez la cellule 798953a6.")

Nombre d'exemples créés : 1000
Aperçu des deux premiers exemples :
 ID: 0, Text: ["A closer look at water-splitting's solar fuel potential"]
 ID: 1, Text: ['An irresistible scent makes locusts swarm, study finds']


In [21]:
model = SentenceTransformer('all-MiniLM-L6-v2')
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)
len(faiss_title_embedding), len(faiss_title_embedding[0])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

(1000, 384)

## 🌟 Exercise 3 · FAISS indexing and search

In [22]:
pdf_to_index = pdf_subset
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)
content_encoded_normalized = faiss_title_embedding.astype('float32')
faiss.normalize_L2(content_encoded_normalized)
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized, id_index)
index_content.ntotal


1000

In [23]:
from sentence_transformers import SentenceTransformer

# Re-initialization of the model in case it was lost from memory
model = SentenceTransformer('all-MiniLM-L6-v2')

def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):
    query_vector = model.encode([query]).astype('float32')
    faiss.normalize_L2(query_vector)
    sims, ids = index_content.search(query_vector, k)
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()
    results['similarities'] = sims[0]
    return results.sort_values(by='similarities', ascending=False)

if 'pdf_to_index' in globals():
    display(search_content('animal', pdf_to_index, k=5))
else:
    print("Erreur : pdf_to_index n'est pas encore défini.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,topic,link,domain,published_date,title,lang,id,similarities
99,TECHNOLOGY,https://www.gematsu.com/2020/08/ghostwire-toky...,gematsu.com,2020-08-07 16:43:13,Ghostwire: Tokyo confirms dog petting,en,99,0.391902
176,TECHNOLOGY,https://www.pushsquare.com/news/2020/08/random...,pushsquare.com,2020-08-03 16:30:00,Random: You Can Pick Up and Pet Cats in Assass...,en,176,0.376784
762,SCIENCE,https://af.reuters.com/article/worldNews/idAFK...,af.reuters.com,2020-08-13 16:51:00,'Secret' life of sharks: Study reveals their s...,en,762,0.344059
928,SCIENCE,https://www.thecut.com/2020/08/scientists-say-...,thecut.com,2020-08-04 12:52:00,Just Let This Lizard Be a Dinosaur,en,928,0.317387
975,HEALTH,https://www.news-medical.net/news/20200813/Res...,news-medical.net,2020-08-13 05:18:00,Researchers explore social behavior of animals...,en,975,0.295497


## 🌟 Exercise 4 · ChromaDB collection and querying

In [3]:
import chromadb
from chromadb.config import Settings

# Utilisation de l'API moderne : PersistentClient permet de sauvegarder les données dans un dossier
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection_name = 'my_news'

# get_or_create_collection est la méthode standard en v0.4+
collection = chroma_client.get_or_create_collection(name=collection_name)

print(f"Collection '{collection_name}' prête.")

Collection 'my_news' prête.


## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [26]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_id = 'google/flan-t5-small'
try:
    # Chargement manuel pour éviter l'erreur de registre de pipeline
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model_t5 = AutoModelForSeq2SeqLM.from_pretrained(model_id)

    query = "What's the latest news on space development?"
    if 'pdf_to_index' in globals() and not pdf_to_index.empty:
        # Recherche du contexte via FAISS
        search_results = search_content(query, pdf_to_index, k=3)
        context = ' '.join(search_results['title'].tolist())

        # Préparation du prompt T5
        input_text = f"answer the question using the context: {context} question: {query}"
        inputs = tokenizer(input_text, return_tensors="pt")

        # Génération
        outputs = model_t5.generate(**inputs, max_length=128)
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        print(f"Question: {query}")
        print(f"Contexte trouvé: {context}")
        print(f"Réponse du modèle: {response}")
    else:
        print("Erreur : pdf_to_index n'est pas disponible. Relancez les cellules précédentes.")
except Exception as e:
    print(f"Erreur lors de la génération : {e}")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Question: What's the latest news on space development?
Contexte trouvé: Orbital space tourism set for rebirth in 2021 Tonight offers best chance of spotting space station in night sky Outrage after NASA ‘goes woke’ and renames ‘insensitive’ space objects
Réponse du modèle: NASA 'goes woke' and renames 'insensitive' space objects
